# 04 - Unsupervised Machine Learning Customer Segmentation (K-Means)
**Platform:** Bangladesh E-Commerce Customer Intelligence Platform  
**Focus:** K-Means Clustering, Feature Scaling, Variance Stabilization, and Cluster Evaluation

---
## Business Rationale
While heuristic RFM segmentation is intuitive, algorithmic clustering discovers multidimensional behavioral boundaries across Recency, Frequency, Monetary, Average Order Value (AOV), and Customer Tenure.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

from src.analytics.eda import load_curated_data
from src.segmentation.rfm import calculate_rfm
from src.segmentation.clustering import prepare_clustering_features, evaluate_clusters, fit_kmeans

df_cust, df_prod, df_ord, df_items = load_curated_data()
df_rfm = calculate_rfm(df_ord)
print(f"Loaded {len(df_rfm):,} customer records.")

## 1. Feature Preprocessing & Scaling
Because financial metrics in e-commerce are heavily right-skewed, we apply a log transform `log1p(x)` followed by `StandardScaler` to ensure unit variance.

In [ ]:
raw_feats, scaled_feats, scaler = prepare_clustering_features(df_rfm)
print("Scaled feature shape:", scaled_feats.shape)

## 2. Cluster Evaluation: Elbow Method & Silhouette Scores

In [ ]:
eval_results = evaluate_clusters(scaled_feats, k_range=range(2, 7))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(eval_results['k_values'], eval_results['inertias'], marker='o', color='#1f77b4', lw=2)
ax1.set_title('Elbow Method (Inertia vs. k)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')

ax2.plot(eval_results['k_values'], eval_results['silhouette_scores'], marker='s', color='#2ca02c', lw=2)
ax2.set_title('Silhouette Score Analysis vs. k', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Coefficient')
plt.tight_layout()
plt.show()

## 3. Fit Optimal K-Means Model (k=4) & Profile Personas

In [ ]:
df_clustered, kmeans_model, profiles = fit_kmeans(df_rfm, scaled_feats, optimal_k=4)
display(profiles)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_clustered,
    x='recency',
    y='monetary',
    hue='cluster_name',
    palette='tab10',
    alpha=0.6
)
plt.yscale('log')
plt.title('Customer Clusters in Recency vs. Monetary Space (Log Scale)', fontsize=13, fontweight='bold')
plt.xlabel('Recency (Days Since Last Purchase)')
plt.ylabel('Monetary Spending (BDT, Log Scale)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()